# AIBL_PrimeKGL
Copied from the original project. Paths are relative to the main folder.
Full PrimeKG embeddings and entity indices are shared by all four cohorts.


In [ ]:
from pathlib import Path
import os
import sys
_starts = [Path.cwd(), Path(globals().get("__file__", Path.cwd() / "notebook")).resolve().parent]
CLINICALKG_ROOT = next(p for start in _starts for p in (start, *start.parents)
                      if (p / "run" / "common" / "paths.py").is_file())
sys.path.insert(0, str(CLINICALKG_ROOT.parent))
from clinicalKG.run.common.paths import PROJECT_ROOT, SHARED_PRIMEKG, ANALYSIS_ROOT, raw_primekg_path, work_dir, asset_path, dataset_dir, result_path
DATASET = 'AIBL'
METHOD = 'PrimeKGL'
WORK_DIR = work_dir(DATASET, METHOD) if METHOD != "CustomKG" else CLINICALKG_ROOT / "run" / "common"
DATASET_DIR = dataset_dir(DATASET)
os.environ.setdefault("KG_ABLATION_OUTPUT", str(ANALYSIS_ROOT))
if DATASET_DIR.is_dir():
    os.chdir(DATASET_DIR)


In [ ]:
# Full PrimeKG is used directly: no seed extraction, blacklist, or Top-k.
RAW_KG_PATH = str(raw_primekg_path())
SHARED_PRIMEKG.mkdir(parents=True, exist_ok=True)
if not Path(RAW_KG_PATH).is_file():
    raise FileNotFoundError(RAW_KG_PATH)
print("Full PrimeKG source:", RAW_KG_PATH)
print("All cohorts share graph-trained assets in:", SHARED_PRIMEKG)


In [ ]:
import pandas as pd
import json
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# 适配 AIBL 的文件路径
AIBL_DATA_PATH = str(DATASET_DIR / 'AIBL诊断.csv')
SEED_CSV_PATH = str(WORK_DIR / 'AIBL_Seed_DementiaHKG.csv')
REFINED_KG_PATH = str(raw_primekg_path())

# 请确认这两个输出文件名是否合适
OUTPUT_GRAPH_PATH = str(SHARED_PRIMEKG / 'PrimeKG-G.csv')
OUTPUT_DICT_PATH = str(WORK_DIR / 'AIBL-PrimeKG-Raw-Entities.json')

MAX_SEQ_LEN = 16 
PAD_TOKEN = "[PAD]"

def build_patient_mapping():
    print("==================================================")
    print("🔗 阶段 3: 构建患者动态映射与客观图谱格式化 (已引入序列填充与注意力掩码)...")

    try:
        kg_df = pd.read_csv(REFINED_KG_PATH)
        triplets_df = kg_df[['x_name', 'relation', 'y_name']].copy()
        triplets_df.columns = ['head', 'relation', 'tail']
        
        triplets_df.to_csv(OUTPUT_GRAPH_PATH, index=False, encoding='utf-8')
        print(f"   ✅ 标准化图谱已生成: {OUTPUT_GRAPH_PATH} (共 {len(triplets_df)} 条边)")
        print("   🔒 [安全检查]: 该图谱中不存在任何患者节点，已彻底阻断 Transductive 泄露。")
    except FileNotFoundError:
        print(f"   ❌ 找不到 {REFINED_KG_PATH}，请确保上一阶段已成功运行。")
        return

    seed_df = pd.read_csv(SEED_CSV_PATH)
    mapping_df = seed_df[seed_df['Category'] == 'Clinical_Mapping']
    aibl_to_pkg_map = dict(zip(mapping_df['AIBL_Feature'], mapping_df['PrimeKG_Node']))
    print(f"\n   📊 成功加载 {len(aibl_to_pkg_map)} 条临床特征映射规则。")

    # 提取核心种子用于优先填充
    core_seeds_df = seed_df[seed_df['Category'].str.contains('Core')]
    core_seeds = core_seeds_df['PrimeKG_Node'].tolist()

    # 替换为 AIBL 的专属阈值
    threshold_rules = {
        "mmse": lambda x: x <= 24,
        "cdr": lambda x: x >= 0.5,
        "apoe": lambda x: x >= 1,
        "lm_imm": lambda x: x <= 8,
        "lm_del": lambda x: x <= 6
    }

    try:
        aibl_df = pd.read_csv(AIBL_DATA_PATH)
    except FileNotFoundError:
        print(f"   ❌ 找不到患者数据 {AIBL_DATA_PATH}。")
        return

    patient_entities = {}
    valid_mapped_patients = 0
    total_features_mapped = 0

    print("\n   ⏳ 正在为每位患者建立固定长度的图谱实体序列与 Attention Mask...")
    
    for index, row in aibl_df.iterrows():
        if 'filename' in row and pd.notna(row['filename']):
            patient_id = str(row['filename']).strip()
        else:
            continue

        active_nodes = []
        
        for col_name, val in row.items():
            if col_name in aibl_to_pkg_map:
                if pd.notna(val) and str(val).strip().lower() not in ['', 'nan', '-4', '-1']:
                    try:
                        num_val = float(val)
                        rule = threshold_rules.get(col_name, lambda x: x > 0)
                        
                        if rule(num_val):
                            primekg_node = aibl_to_pkg_map[col_name]
                            active_nodes.append(primekg_node)
                            total_features_mapped += 1
                    except ValueError:
                        pass

        active_nodes = list(set(active_nodes))
        
        # 优先使用医学核心种子填补空白
        core_idx = 0
        while len(active_nodes) < MAX_SEQ_LEN and core_idx < len(core_seeds):
            if core_seeds[core_idx] not in active_nodes:
                active_nodes.append(core_seeds[core_idx])
            core_idx += 1

        seq_len = len(active_nodes)
        
        if seq_len > MAX_SEQ_LEN:
            active_nodes = active_nodes[:MAX_SEQ_LEN]
            attention_mask = [1] * MAX_SEQ_LEN
        else:
            pad_length = MAX_SEQ_LEN - seq_len
            attention_mask = [1] * seq_len + [0] * pad_length
            active_nodes = active_nodes + [PAD_TOKEN] * pad_length

        patient_entities[patient_id] = {
            "input_nodes": active_nodes,
            "attention_mask": attention_mask
        }
        
        if seq_len > 0:
            valid_mapped_patients += 1

    with open(OUTPUT_DICT_PATH, 'w', encoding='utf-8') as f:
        json.dump(patient_entities, f, indent=4, ensure_ascii=False)

    print("\n==================================================")
    print("💾 阶段 3 任务完成！")
    print(f"   👨‍⚕️ 处理患者总数: {len(aibl_df)}")
    print(f"   🔗 成功为 {valid_mapped_patients} 名患者提取了图谱实体。")
    print(f"   📈 总计完成特征到实体的映射: {total_features_mapped} 次。")
    print(f"   📏 序列统一长度 (MAX_SEQ_LEN): {MAX_SEQ_LEN}")
    print(f"   📄 患者动态查询字典 (包含 Mask) 已保存至: {OUTPUT_DICT_PATH}")
    print("==================================================")

if __name__ == "__main__":
    build_patient_mapping()

In [ ]:
# Same DistMult score/loss; one shared full-graph pretraining job.
from clinicalKG.PrimeKGL.full_primekg import train_full_primekg, plot_loss
RESUME = False  # Set True to continue PrimeKG-DistMult-last.pt.
NUM_EPOCHS = 50  # Total target epochs, shared across cohorts.
EMBED_DIM = 128
BATCH_SIZE = 512
LR = 0.005
training_state = train_full_primekg(epochs=NUM_EPOCHS, embed_dim=EMBED_DIM,
                                   batch_size=BATCH_SIZE, lr=LR, resume=RESUME)
plot_loss()


In [ ]:
import matplotlib.pyplot as plt
import json
import os

# ================= 配置区 =================
LOSS_HISTORY_PATH = asset_path(DATASET, 'PrimeKG', 'AIBL-PrimeKG-Raw-LossHistory.json')
OUTPUT_PLOT_PATH = asset_path(DATASET, 'PrimeKG', 'AIBL-PrimeKG-Raw-LossCurve.png')

def plot_loss_curve():
    print("==================================================")
    print("📈 步骤 4: 正在绘制 KGE 训练 Loss 收敛曲线...")
    
    if not os.path.exists(LOSS_HISTORY_PATH):
        print(f"❌ 找不到 Loss 记录文件: {LOSS_HISTORY_PATH}，请确认上一步已成功训练。")
        return
        
    with open(LOSS_HISTORY_PATH, 'r', encoding='utf-8') as f:
        losses = json.load(f)
        
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(losses) + 1), losses, color='#2c3e50', linewidth=2, label='DistMult Train Loss')
    
    # 图表美化设置
    plt.title('DementiaHKG Knowledge Graph Embedding Loss', fontsize=16, fontweight='bold')
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('Margin Ranking Loss', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(fontsize=12)
    plt.tight_layout()
    
    # 保存高斯清晰度的图片供论文使用
    plt.savefig(OUTPUT_PLOT_PATH, dpi=300)
    print(f"   ✅ 曲线图已生成，正在当前 Cell 渲染，并已保存高清大图至: {OUTPUT_PLOT_PATH}")
    print("==================================================")
    
    plt.show()

if __name__ == "__main__":
    plot_loss_curve()